# Data Cleaning Pipeline
**Dissertation:** AI Approaches to Analysing Recruitment Demand: Machine learning insights from European Pharmaceutical Job Postings  
**Author:** Kashmira Bhoir  
**Institution:**  GISMA University of Applied Sciences  
**Date:** 21 June 2026

## Overview
This notebook helps to clean and standardise the combined pharmaceutical job postings dataset making it ready for RQ1, RQ2 and RQ3 analysis.

## Input
- `Europe_Pharma_Job_Postings Combined Raw.xlsx` — raw combined dataset (30,516 rows)

## Output
- `Europe_Pharma_Jobs_Cleaned.xlsx` — cleaned dataset (9,520 rows)

## Cleaning steps are as follows
1. Standardised column names
2. Normalising the post_date
3. Removing unparseable post_date rows
4. Removing duplicate postings
5. Cleaning Text columns
6. Fix category 'Nan' artifact and label missing categories
7. Converting Salaries to EUROS (year-specific rates, hourly->annually)
8. Standardising Job_type
9. Standardising location -> country
10. Filtering pharma specific job postings
11. Saved Final cleaned dataset


Install and Import Libraries

In [ ]:
!pip install pandas numpy -q

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")


Libraries loaded


Upload .csv file

In [ ]:
file_path = "/content/Europe_Pharma_Job_Postings Combined Raw.xlsx"
df = pd.read_excel(file_path, sheet_name="Job Postings")

print(f"Loaded: {df.shape[0]:,} rows  |  {df.shape[1]} columns")
print(f"\n  Raw columns: {df.columns.tolist()}")

Loaded: 30,516 rows  |  9 columns

  Raw columns: ['Data_Source', 'Category', 'Company_name', 'Job_description', 'Job_title', 'Job_type', 'Location', 'Post_date', 'Salary_offered']


Standardised column names

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
print("Column names standardised:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

Column names standardised:
   1. data_source
   2. category
   3. company_name
   4. job_description
   5. job_title
   6. job_type
   7. location
   8. post_date
   9. salary_offered


Normalising the post_date column

In [ ]:
df['post_date'] = pd.to_datetime(df['post_date'], errors='coerce')

print(f"post_date dtype: {df['post_date'].dtype}")
print(f"Unparseable dates (now NaT): {df['post_date'].isnull().sum()}")

post_date dtype: datetime64[ns]
Unparseable dates (now NaT): 3


Dropping the unparseable post_date rows

In [ ]:
before = len(df)
df = df[df['post_date'].notna()]
after = len(df)

print(f"Dropped {before - after} rows with unparseable post_date")
print(f"Remaining Rows: {after:,}")

Dropped 3 rows with unparseable post_date
Remaining Rows: 30,513


Removed duplicates

In [ ]:
before = len(df)
df.drop_duplicates(
    subset   = ['job_title', 'company_name', 'location', 'post_date', 'job_description'],
    inplace  = True
)
after = len(df)

print(f"Duplicates removed : {before - after:,}")
print(f"   Rows remaining     : {after:,}")

Duplicates removed : 20,848
   Rows remaining     : 9,665


Standardising Text Columns

In [ ]:
text_cols = ['job_title', 'company_name', 'category', 'location', 'job_type']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

print("Text columns cleaned")
print(f"\n  Sample job titles:")
print(df['job_title'].value_counts().head(5).to_string())

Text columns cleaned

  Sample job titles:
job_title
Medical Sales Representative    222
Account Manager                 215
Key Account Manager             193
Territory Manager               151
Senior Medical Writer           121


Fix category 'Nan' artifact and label missing categories

In [ ]:
df['category'] = df['category'].replace('Nan', None)
df['category'] = df['category'].fillna('Uncategorized')

print(f"Rows now labelled 'Uncategorized': {(df['category'] == 'Uncategorized').sum():,}")
print(f"Category nulls remaining: {df['category'].isnull().sum()}")

Rows now labelled 'Uncategorized': 189
Category nulls remaining: 0


Handled currency rates, location fallback and the conversion functions

In [ ]:
EUR_RATES = {
    2018: {'GBP': 1.1364, 'CHF': 0.8621, 'CZK': 0.03898, 'USD': 0.8475, 'EUR': 1.0},
    2021: {'GBP': 1.1634, 'CHF': 0.9259, 'CZK': 0.0390,  'USD': 0.8475, 'EUR': 1.0},
    2026: {'GBP': 1.1541, 'CHF': 1.0813, 'CZK': 0.04118, 'USD': 0.8721, 'EUR': 1.0},
}

LOCATION_CURRENCY = {
    'united kingdom': 'GBP', 'south east': 'GBP', 'north west': 'GBP',
    'switzerland': 'CHF',
    'germany': 'EUR', 'france': 'EUR', 'italy': 'EUR', 'spain': 'EUR',
    'ireland': 'EUR', 'netherlands': 'EUR', 'austria': 'EUR',
    'portugal': 'EUR', 'belgium': 'EUR', 'denmark': 'EUR',
    'europe': 'GBP',
}

HOURS_PER_YEAR = 2080

def detect_currency(val_low):
    if re.search(r'chf|swiss\s*franc', val_low): return 'CHF'
    if re.search(r'czk', val_low):                return 'CZK'
    if re.search(r'us\$|usd', val_low):           return 'USD'
    if re.search(r'£|gbp', val_low):              return 'GBP'
    if re.search(r'€|eur', val_low):              return 'EUR'
    if re.search(r'\$', val_low):                 return 'USD'
    return None

def parse_salary_eur(val, location, year):
    if pd.isna(val): return None, None, None
    val_low = str(val).strip().lower()
    val_low = re.sub(r'(\d+)k(\d)\b', r'\1\2k', val_low)

    vague = ['competitive','negotiable','neg ','on application',
             'tbc','doe','excellent','attractive','discussed',
             'depending on','uncapped','market rate']
    if any(v in val_low for v in vague): return None, None, None
    if not re.search(r'\d', val_low):    return None, None, None

    currency = detect_currency(val_low)
    if currency is None:
        currency = LOCATION_CURRENCY.get(str(location).strip().lower())
    if currency is None:
        return None, None, None

    is_hourly = bool(re.search(r'p\s*hour|per\s*hour|/hr', val_low))

    s = re.sub(r'swiss\s*franc|chf|czk|us\$|usd|gbp|eur|€|£|\$', '', val_low)
    s = re.sub(r'up\s*to|approx|from|starting\s*at', '', s)
    s = re.sub(r'(pa|per\s*annum|p\.a\.|pm|per\s*month|p\s*hour|per\s*hour|/hr).*$', '', s)
    s = re.sub(r'(\d),(\d)', r'\1\2', s)
    s = re.sub(r'\+.*$', '', s).strip()

    has_k = bool(re.search(r'\d\s*k', val_low))
    numbers = re.findall(r'\d+\.?\d*', s)
    if not numbers: return None, None, None

    nums = [float(n) * (1000 if has_k and float(n) < 1000 else 1) for n in numbers]

    if is_hourly:
        nums = [n * HOURS_PER_YEAR for n in nums]

    rate = EUR_RATES.get(int(year), {}).get(currency) if pd.notna(year) else None
    if rate is None: return None, None, None
    nums = [n * rate for n in nums]

    nums = [n for n in nums if 10000 <= n <= 500000]
    if not nums: return None, None, currency
    if len(nums) >= 2: return round(nums[0], 2), round(nums[1], 2), currency
    return round(nums[0], 2), round(nums[0], 2), currency

Converting Salaries to EUROS

In [ ]:
df['post_year'] = df['post_date'].dt.year

results = df.apply(
    lambda r: pd.Series(parse_salary_eur(r['salary_offered'], r['location'], r['post_year'])),
    axis=1
)
df[['salary_min_eur', 'salary_max_eur', 'currency_detected']] = results

df['salary_mid_eur'] = df[['salary_min_eur', 'salary_max_eur']].mean(axis=1)

print(f"Rows with a usable EUR salary: {df['salary_min_eur'].notna().sum():,} of {len(df):,}")
print(f"\nCurrency breakdown (parsed rows):")
print(df.loc[df['salary_min_eur'].notna(), 'currency_detected'].value_counts())

print(f"\nSample conversions:")
print(df[df['salary_min_eur'].notna()][
    ['salary_offered','location','post_year','currency_detected','salary_min_eur','salary_max_eur','salary_mid_eur']
].head(8).to_string(index=False))

Rows with a usable EUR salary: 1,736 of 9,665

Currency breakdown (parsed rows):
currency_detected
GBP    1357
EUR     193
CHF     170
USD      16
Name: count, dtype: int64

Sample conversions:
                        salary_offered       location  post_year currency_detected  salary_min_eur  salary_max_eur  salary_mid_eur
                          60k - 70k pa        Germany       2018               EUR         60000.0         70000.0        65000.00
                          Up to 80k pa         France       2018               EUR         80000.0         80000.0        80000.00
                    CHF 95,000-110,000    Switzerland       2018               CHF         81899.5         94831.0        88365.25
   Swiss Franc80k - Swiss Franc110k pa    Switzerland       2018               CHF         68968.0         94831.0        81899.50
                               45,000+ United Kingdom       2018               GBP         51138.0         51138.0        51138.00
                    

Standardising Job Type

In [ ]:
job_type_map = {
    'permanent': 'Full-Time', 'full-time': 'Full-Time', 'full time': 'Full-Time',
    'contract/interim': 'Contract', 'contract/temp': 'Contract',
    'contract': 'Contract', 'contractor': 'Contract',
    'part-time': 'Part-Time', 'part time': 'Part-Time',
    'temporary/seasonal': 'Temporary', 'temporary': 'Temporary',
    'remote': 'Remote', 'internship': 'Internship', 'any': 'Other',
}

df['job_type_clean'] = df['job_type'].str.lower().map(job_type_map).fillna('Other')

print("Job types standardised:")
print(df['job_type_clean'].value_counts())

Job types standardised:
job_type_clean
Full-Time     8619
Contract       874
Other           82
Temporary       53
Part-Time       34
Internship       3
Name: count, dtype: int64


Standardising Location

In [ ]:
location_map = {
    'united kingdom': 'United Kingdom',
    'south east':      'United Kingdom',
    'north west':      'United Kingdom',
    'europe':          'Europe (General)',
    'switzerland':     'Switzerland',
    'germany':         'Germany',
    'france':          'France',
    'italy':           'Italy',
    'spain':           'Spain',
    'ireland':         'Ireland',
    'denmark':         'Denmark',
    'portugal':        'Portugal',
    'poland':          'Poland',
    'austria':         'Austria',
    'netherlands':     'Netherlands',
    'belgium':         'Belgium',
    'sweden':          'Sweden',
}

df['country'] = df['location'].str.lower().map(location_map).fillna('Other')

print("Standardising Location:")
print(df['country'].value_counts())

Standardising Location:
country
United Kingdom      5272
Europe (General)    3389
Switzerland          354
Germany              335
France               119
Spain                 81
Italy                 78
Ireland               20
Denmark                7
Poland                 2
Portugal               2
Austria                2
Netherlands            2
Belgium                1
Sweden                 1
Name: count, dtype: int64


Filtering only pharma jobs

In [ ]:
pharma_keywords = [
    'Pharmacovigilance', 'Regulatory Affairs', 'Clinical Research',
    'Drug Safety', 'GMP', 'Biotech', 'Pharmaceutical', 'Medical Affairs',
    'Quality Assurance', 'Quality Control', 'Drug Discovery',
    'Clinical Trials', 'Medical Writing', 'Validation', 'Science',
    'Manufacturing', 'Data Management', 'Account Manager', 'Medical Sales'
]
pattern = '|'.join(pharma_keywords)

search_text = (
    df['job_title'] + ' ' +
    df['category'].fillna('') + ' ' +
    df['job_description']
).str.replace('-', ' ', regex=False)

mask = search_text.str.contains(pattern, case=False, na=False)

before = len(df)
df = df[mask].copy()
after = len(df)

print(f"Pharma filter applied")
print(f"   Before : {before:,}")
print(f"   After  : {after:,}")
print(f"   Removed: {before - after:,} non-pharma rows")

Pharma filter applied
   Before : 9,665
   After  : 9,520
   Removed: 145 non-pharma rows


Summary of Final Cleaned Dataset

In [ ]:

print("  FINAL CLEANED DATASET")
print(f"\n  Rows    : {len(df):,}")
print(f"  Columns : {df.shape[1]}")

print(f"\n  Missing values:")
for col in df.columns:
    nulls = df[col].isnull().sum()
    pct   = (nulls / len(df)) * 100
    flag  = "Info" if pct > 10 else "Success"
    print(f"  {flag} {col:<25} {nulls:>6,}  ({pct:.1f}%)")

print(f"\n  Sample cleaned data (key columns):")
print(df[['job_title', 'category', 'country',
          'job_type_clean', 'salary_offered',
          'salary_min_eur', 'salary_max_eur']].head(5).to_string(index=False))

  FINAL CLEANED DATASET

  Rows    : 9,520
  Columns : 16

  Missing values:
  Success data_source                    0  (0.0%)
  Success category                       0  (0.0%)
  Success company_name                   0  (0.0%)
  Success job_description                0  (0.0%)
  Success job_title                      0  (0.0%)
  Success job_type                       0  (0.0%)
  Success location                       0  (0.0%)
  Success post_date                      0  (0.0%)
  Success salary_offered                 2  (0.0%)
  Success post_year                      0  (0.0%)
  Info salary_min_eur             7,840  (82.4%)
  Info salary_max_eur             7,840  (82.4%)
  Info currency_detected          5,737  (60.3%)
  Info salary_mid_eur             7,840  (82.4%)
  Success job_type_clean                 0  (0.0%)
  Success country                        0  (0.0%)

  Sample cleaned data (key columns):
                              job_title                       category       

Saving Cleaned file

In [ ]:
output_path = "/content/Europe_Pharma_Jobs_Cleaned.xlsx"
df.to_excel(output_path, index=False)

print(f"Cleaned file saved → {output_path}")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {df.shape[1]}")
print(f"\n   Columns in cleaned file:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

Cleaned file saved → /content/Europe_Pharma_Jobs_Cleaned.xlsx
   Rows    : 9,520
   Columns : 16

   Columns in cleaned file:
   1. data_source
   2. category
   3. company_name
   4. job_description
   5. job_title
   6. job_type
   7. location
   8. post_date
   9. salary_offered
   10. post_year
   11. salary_min_eur
   12. salary_max_eur
   13. currency_detected
   14. salary_mid_eur
   15. job_type_clean
   16. country
